# 3. Feature Engineering
Point-in-time feature engineering after time-series split.

In [2]:
"""Point-in-time feature engineering after time-series split.

Design
------
This script must run AFTER:

    01_split_time_series_data.py

It reads the split parquet files from ``data/model/v3`` and writes feature
parquet files to ``data/model/v3/features``.

Leakage rule
------------
All target-derived features use only timestamps strictly before the row being
scored:

    lag_k              = target.shift(k)
    rolling_window     = target.shift(1).rolling(window)

Validation and test are allowed to look backward into earlier splits for
history. They are not allowed to look at their own current/future target.

Examples:

* fold validation features are built from ``fold_train + fold_val`` and then
  only fold_val rows are exported.
* final test features are built from ``development + test`` and then only test
  rows are exported.

This is the practical forecasting pattern:

    split first -> feature engineering with backward context -> train/evaluate
"""

from __future__ import annotations

import argparse
import json
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd


PROJECT_ROOT = Path.cwd().resolve().parents[3]
DEFAULT_SPLIT_DIR = PROJECT_ROOT / "data" / "model" / "v3"
DEFAULT_OUTPUT_DIR = DEFAULT_SPLIT_DIR / "features"

SITE_COL = "site_id"
TIMESTAMP_COL = "timestamp"
TARGET_COL = "energy_generated_kwh"

SOUTHERN_SEASON_MAP: dict[int, str] = {
    12: "summer",
    1: "summer",
    2: "summer",
    3: "autumn",
    4: "autumn",
    5: "autumn",
    6: "winter",
    7: "winter",
    8: "winter",
    9: "spring",
    10: "spring",
    11: "spring",
}

SEASON_CODE_MAP: dict[str, int] = {
    "summer": 0,
    "autumn": 1,
    "winter": 2,
    "spring": 3,
}

DEFAULT_LAGS: tuple[int, ...] = (1, 4, 96)
DEFAULT_ROLLING_WINDOWS: tuple[int, ...] = (4, 12, 96)
DEFAULT_CATEGORICAL_COLS: tuple[str, ...] = (
    "site_id",
    "campus_name",
    "location_name",
    "site_metric",
    "panel",
    "inverter",
    "optimizers",
    "weather_join_method",
    "weather_condition",
    "weather_description",
)


@dataclass(frozen=True)
class TimeFeatureConfig:
    """Configuration for split-aware time-series feature engineering."""

    split_dir: Path = DEFAULT_SPLIT_DIR
    output_dir: Path = DEFAULT_OUTPUT_DIR
    version: str = "v3"
    expected_freq_minutes: int = 15
    lags: tuple[int, ...] = DEFAULT_LAGS
    rolling_windows: tuple[int, ...] = DEFAULT_ROLLING_WINDOWS
    categorical_cols: tuple[str, ...] = DEFAULT_CATEGORICAL_COLS
    timestamp_col: str = TIMESTAMP_COL
    site_col: str = SITE_COL
    target_col: str = TARGET_COL


def require_columns(df: pd.DataFrame, columns: Iterable[str]) -> None:
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")


def parse_int_tuple(value: str) -> tuple[int, ...]:
    return tuple(int(x.strip()) for x in value.split(",") if x.strip())


def parse_str_tuple(value: str) -> tuple[str, ...]:
    return tuple(x.strip() for x in value.split(",") if x.strip())


def read_parquet(path: Path, config: TimeFeatureConfig) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Required split parquet not found: {path}")
    df = pd.read_parquet(path)
    require_columns(df, [config.site_col, config.timestamp_col, config.target_col])
    df[config.timestamp_col] = pd.to_datetime(df[config.timestamp_col], errors="coerce")
    return df.sort_values([config.site_col, config.timestamp_col]).reset_index(drop=True)


def write_parquet(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)


def add_time_features(df: pd.DataFrame, config: TimeFeatureConfig) -> pd.DataFrame:
    """Add calendar/cyclical features from timestamp only."""

    out = df.copy()
    ts = pd.to_datetime(out[config.timestamp_col], errors="coerce")
    minute_of_day = ts.dt.hour * 60 + ts.dt.minute
    day_of_year = ts.dt.dayofyear

    out[f"{config.version}_minute_of_day"] = minute_of_day
    out[f"{config.version}_hour_sin"] = np.sin(2 * np.pi * minute_of_day / 1440.0)
    out[f"{config.version}_hour_cos"] = np.cos(2 * np.pi * minute_of_day / 1440.0)
    out[f"{config.version}_doy_sin"] = np.sin(2 * np.pi * day_of_year / 365.25)
    out[f"{config.version}_doy_cos"] = np.cos(2 * np.pi * day_of_year / 365.25)
    out[f"{config.version}_month"] = ts.dt.month
    out[f"{config.version}_day_of_week"] = ts.dt.dayofweek
    out[f"{config.version}_is_weekend"] = (
        out[f"{config.version}_day_of_week"].isin([5, 6]).astype("int8")
    )
    out[f"{config.version}_season"] = (
        out[f"{config.version}_month"].map(SOUTHERN_SEASON_MAP).astype("string")
    )
    out[f"{config.version}_season_code"] = (
        out[f"{config.version}_season"].map(SEASON_CODE_MAP).astype("Int64")
    )

    # Some marts use hour=-1 to represent the previous-hour bucket. For model
    # timestamp features, keep both raw and model-safe version.
    if "hour" in out.columns:
        out[f"{config.version}_hour_bucket_raw"] = pd.to_numeric(
            out["hour"], errors="coerce"
        )
        out[f"{config.version}_hour_bucket_model"] = out[
            f"{config.version}_hour_bucket_raw"
        ].replace(-1, 23)

    return out


def add_metadata_features(df: pd.DataFrame, config: TimeFeatureConfig) -> pd.DataFrame:
    """Add non-target numeric metadata features and missing flags.

    Categorical encoding is handled after split-specific feature generation so
    train/development fit maps are reused for val/test. Do not call
    ``cat.codes`` here; that can assign inconsistent codes across folds.
    """

    out = df.copy()

    for col in ("capacity_kw", "number_of_panels"):
        if col in out.columns:
            out[f"{config.version}_{col}_missing_flag"] = out[col].isna().astype("int8")

    if {"capacity_kw", "number_of_panels"}.issubset(out.columns):
        out[f"{config.version}_capacity_per_panel"] = (
            out["capacity_kw"] / out["number_of_panels"]
        )
        out.loc[
            ~np.isfinite(out[f"{config.version}_capacity_per_panel"]),
            f"{config.version}_capacity_per_panel",
        ] = np.nan

    return out


def existing_categorical_cols(
    df: pd.DataFrame,
    config: TimeFeatureConfig,
) -> list[str]:
    """Return configured categorical columns that exist in this dataframe."""

    return [col for col in config.categorical_cols if col in df.columns]


def fit_category_maps(
    train_df: pd.DataFrame,
    config: TimeFeatureConfig,
) -> dict[str, dict[str, int]]:
    """Fit stable ordinal maps on train-side data only.

    Mapping convention:
        missing value -> 0
        known category -> 1..N
        unseen value at transform time -> -1
    """

    maps: dict[str, dict[str, int]] = {}
    for col in existing_categorical_cols(train_df, config):
        values = (
            train_df[col]
            .astype("string")
            .fillna("__MISSING__")
            .drop_duplicates()
            .sort_values()
            .tolist()
        )
        mapping: dict[str, int] = {"__MISSING__": 0}
        next_code = 1
        for value in values:
            if value == "__MISSING__":
                continue
            mapping[value] = next_code
            next_code += 1
        maps[col] = mapping
    return maps


def apply_category_maps(
    df: pd.DataFrame,
    maps: dict[str, dict[str, int]],
    config: TimeFeatureConfig,
) -> pd.DataFrame:
    """Transform categorical columns with train-fitted maps."""

    out = df.copy()
    for col, mapping in maps.items():
        if col not in out.columns:
            continue
        values = out[col].astype("string").fillna("__MISSING__")
        encoded = values.map(mapping).fillna(-1).astype("int32")
        out[f"{config.version}_{col}_enc"] = encoded
        out[f"{config.version}_{col}_unknown_flag"] = encoded.eq(-1).astype("int8")
    return out


def encode_train_and_other(
    *,
    train_df: pd.DataFrame,
    other_df: pd.DataFrame | None,
    config: TimeFeatureConfig,
) -> tuple[pd.DataFrame, pd.DataFrame | None, dict[str, dict[str, int]]]:
    """Fit category maps on train_df, transform train_df and optional other_df."""

    maps = fit_category_maps(train_df, config)
    train_encoded = apply_category_maps(train_df, maps, config)
    other_encoded = apply_category_maps(other_df, maps, config) if other_df is not None else None
    return train_encoded, other_encoded, maps


def add_weather_domain_features(
    df: pd.DataFrame, config: TimeFeatureConfig
) -> pd.DataFrame:
    """Add exogenous weather/domain interactions."""

    out = df.copy()
    prefix = config.version

    if {"shortwave_radiation", "temperature_c"}.issubset(out.columns):
        out[f"{prefix}_temp_x_shortwave"] = (
            out["temperature_c"] * out["shortwave_radiation"]
        )

    if {"shortwave_radiation", "diffuse_solar_radiation"}.issubset(out.columns):
        denom = out["shortwave_radiation"].replace(0, np.nan)
        out[f"{prefix}_diffuse_ratio"] = (
            out["diffuse_solar_radiation"] / denom
        ).clip(lower=0, upper=10)

    if {"direct_normal_irradiance", "shortwave_radiation"}.issubset(out.columns):
        denom = out["shortwave_radiation"].replace(0, np.nan)
        out[f"{prefix}_dni_ratio"] = (
            out["direct_normal_irradiance"] / denom
        ).clip(lower=0, upper=10)

    if {"cloud_cover_total", "shortwave_radiation"}.issubset(out.columns):
        out[f"{prefix}_cloud_x_shortwave"] = (
            out["cloud_cover_total"] * out["shortwave_radiation"]
        )

    return out


def continuous_history_mask(
    group: pd.DataFrame,
    *,
    timestamp_col: str,
    window_steps: int,
    expected_freq_minutes: int,
) -> pd.Series:
    """True when the previous ``window_steps`` intervals are contiguous."""

    diffs = group[timestamp_col].diff().dt.total_seconds().div(60.0)
    is_expected_gap = diffs.eq(expected_freq_minutes)
    return (
        is_expected_gap.rolling(window_steps, min_periods=window_steps)
        .sum()
        .eq(window_steps)
        .fillna(False)
    )


def add_lag_rolling_features(
    df: pd.DataFrame,
    config: TimeFeatureConfig,
) -> pd.DataFrame:
    """Add leakage-safe target lag/rolling features.

    Every target-derived feature is shifted. Rolling windows use shifted target.
    Feature values are nulled if the historical window crosses a time gap.
    """

    out = df.copy()
    out = out.sort_values([config.site_col, config.timestamp_col]).reset_index(drop=True)
    grouped = out.groupby(config.site_col, group_keys=False, observed=True)
    prefix = config.version

    for lag in config.lags:
        col = f"{prefix}_lag_{lag}"
        out[col] = grouped[config.target_col].shift(lag)
        valid = grouped.apply(
            continuous_history_mask,
            timestamp_col=config.timestamp_col,
            window_steps=lag,
            expected_freq_minutes=config.expected_freq_minutes,
        )
        out.loc[~valid.to_numpy(), col] = np.nan

    shifted_target = grouped[config.target_col].shift(1)

    for window in config.rolling_windows:
        valid = grouped.apply(
            continuous_history_mask,
            timestamp_col=config.timestamp_col,
            window_steps=window,
            expected_freq_minutes=config.expected_freq_minutes,
        ).to_numpy()

        rolling = shifted_target.groupby(out[config.site_col]).rolling(
            window, min_periods=window
        )
        feature_map = {
            f"{prefix}_rolling_mean_{window}": rolling.mean(),
            f"{prefix}_rolling_std_{window}": rolling.std(),
            f"{prefix}_rolling_min_{window}": rolling.min(),
            f"{prefix}_rolling_max_{window}": rolling.max(),
        }
        for col, values in feature_map.items():
            out[col] = values.reset_index(level=0, drop=True)
            out.loc[~valid, col] = np.nan

    target_feature_cols = [
        col
        for col in out.columns
        if col.startswith(f"{prefix}_lag_")
        or col.startswith(f"{prefix}_rolling_")
    ]
    out[f"{prefix}_has_complete_history_features"] = ~out[
        target_feature_cols
    ].isna().any(axis=1)
    return out


def build_features_with_backward_context(
    *,
    context_df: pd.DataFrame | None,
    target_df: pd.DataFrame,
    config: TimeFeatureConfig,
    output_role: str,
) -> pd.DataFrame:
    """Build features for target rows with optional earlier historical context."""

    target = target_df.copy()
    target["_feature_export_row"] = True
    target["_feature_output_role"] = output_role

    frames = []
    if context_df is not None and len(context_df):
        context = context_df.copy()
        context["_feature_export_row"] = False
        context["_feature_output_role"] = "history_context"
        frames.append(context)
    frames.append(target)

    combined = pd.concat(frames, ignore_index=True, sort=False)
    combined[config.timestamp_col] = pd.to_datetime(
        combined[config.timestamp_col], errors="coerce"
    )
    combined = combined.sort_values([config.site_col, config.timestamp_col]).reset_index(
        drop=True
    )

    features = add_time_features(combined, config)
    features = add_metadata_features(features, config)
    features = add_weather_domain_features(features, config)
    features = add_lag_rolling_features(features, config)

    export = features[features["_feature_export_row"].astype(bool)].copy()
    export = export.drop(columns=["_feature_export_row"])
    return export.reset_index(drop=True)


def summarize_features(
    *,
    name: str,
    df: pd.DataFrame,
    config: TimeFeatureConfig,
) -> dict[str, object]:
    prefix = config.version
    feature_cols = [col for col in df.columns if col.startswith(f"{prefix}_")]
    row: dict[str, object] = {
        "name": name,
        "rows": int(len(df)),
        "columns": int(df.shape[1]),
        "feature_columns": int(len(feature_cols)),
        "site_count": int(df[config.site_col].nunique()) if len(df) else 0,
        "min_timestamp": df[config.timestamp_col].min() if len(df) else pd.NaT,
        "max_timestamp": df[config.timestamp_col].max() if len(df) else pd.NaT,
    }
    complete_col = f"{prefix}_has_complete_history_features"
    if complete_col in df.columns:
        row["complete_history_rows"] = int(df[complete_col].fillna(False).sum())
        row["incomplete_history_rows"] = int((~df[complete_col].fillna(False)).sum())
    encoded_cols = [col for col in df.columns if col.startswith(f"{prefix}_") and col.endswith("_enc")]
    unknown_cols = [
        col
        for col in df.columns
        if col.startswith(f"{prefix}_") and col.endswith("_unknown_flag")
    ]
    row["encoded_categorical_columns"] = int(len(encoded_cols))
    row["unknown_category_rows_total"] = int(df[unknown_cols].sum().sum()) if unknown_cols else 0
    for col in (f"{prefix}_lag_1", f"{prefix}_lag_4", f"{prefix}_rolling_mean_4"):
        if col in df.columns:
            row[f"{col}_null_rows"] = int(df[col].isna().sum())
    return row


def feature_paths(config: TimeFeatureConfig) -> dict[str, Path]:
    version = config.version
    return {
        "development": config.output_dir / "development" / f"{version}_development_features.parquet",
        "test": config.output_dir / "test" / f"{version}_test_features.parquet",
        "final_train": config.output_dir / "final_train" / f"{version}_final_train_features.parquet",
        "train_alias": config.output_dir / "train" / f"{version}_train_features.parquet",
        "val_alias": config.output_dir / "val" / f"{version}_val_features.parquet",
        "summary": config.output_dir / "summaries" / f"{version}_feature_summary.csv",
        "config": config.output_dir / "summaries" / f"{version}_feature_config.json",
        "category_maps": config.output_dir
        / "summaries"
        / f"{version}_category_encoding_maps.json",
    }


def run_feature_engineering(config: TimeFeatureConfig) -> dict[str, Path]:
    """Generate point-in-time feature files for all split outputs."""

    split = config.split_dir
    version = config.version

    development = read_parquet(
        split / "development" / f"{version}_development.parquet", config
    )
    test = read_parquet(split / "test" / f"{version}_test.parquet", config)
    train_alias = read_parquet(split / "train" / f"{version}_train.parquet", config)
    val_alias = read_parquet(split / "val" / f"{version}_val.parquet", config)

    paths = feature_paths(config)
    for path in paths.values():
        path.parent.mkdir(parents=True, exist_ok=True)

    development_features = build_features_with_backward_context(
        context_df=None,
        target_df=development,
        config=config,
        output_role="development",
    )
    final_train_features = development_features.copy()
    train_features = build_features_with_backward_context(
        context_df=None,
        target_df=train_alias,
        config=config,
        output_role="train",
    )
    val_features = build_features_with_backward_context(
        context_df=train_alias,
        target_df=val_alias,
        config=config,
        output_role="val",
    )
    test_features = build_features_with_backward_context(
        context_df=development,
        target_df=test,
        config=config,
        output_role="test",
    )

    category_maps: dict[str, dict[str, dict[str, int]]] = {}
    development_features, test_features, development_maps = encode_train_and_other(
        train_df=development_features,
        other_df=test_features,
        config=config,
    )
    final_train_features = apply_category_maps(
        final_train_features,
        development_maps,
        config,
    )
    category_maps["development_to_test"] = development_maps

    train_features, val_features, train_alias_maps = encode_train_and_other(
        train_df=train_features,
        other_df=val_features,
        config=config,
    )
    category_maps["train_alias_to_val_alias"] = train_alias_maps

    write_parquet(development_features, paths["development"])
    write_parquet(final_train_features, paths["final_train"])
    write_parquet(train_features, paths["train_alias"])
    write_parquet(val_features, paths["val_alias"])
    write_parquet(test_features, paths["test"])

    summary_rows = [
        summarize_features(name="development", df=development_features, config=config),
        summarize_features(name="final_train", df=final_train_features, config=config),
        summarize_features(name="train_alias", df=train_features, config=config),
        summarize_features(name="val_alias_backward_from_train", df=val_features, config=config),
        summarize_features(name="test_backward_from_development", df=test_features, config=config),
    ]

    folds_dir = split / "time_series_folds"
    feature_folds_dir = config.output_dir / "time_series_folds"
    fold_train_paths = sorted(folds_dir.glob("fold_*_train.parquet"))
    for fold_train_path in fold_train_paths:
        fold_name = fold_train_path.stem.replace("_train", "")
        fold_val_path = folds_dir / f"{fold_name}_val.parquet"
        if not fold_val_path.exists():
            raise FileNotFoundError(f"Missing fold validation parquet: {fold_val_path}")

        fold_train = read_parquet(fold_train_path, config)
        fold_val = read_parquet(fold_val_path, config)
        fold_train_features = build_features_with_backward_context(
            context_df=None,
            target_df=fold_train,
            config=config,
            output_role=f"{fold_name}_train",
        )
        fold_val_features = build_features_with_backward_context(
            context_df=fold_train,
            target_df=fold_val,
            config=config,
            output_role=f"{fold_name}_val",
        )
        fold_train_features, fold_val_features, fold_maps = encode_train_and_other(
            train_df=fold_train_features,
            other_df=fold_val_features,
            config=config,
        )
        category_maps[f"{fold_name}_train_to_val"] = fold_maps
        write_parquet(
            fold_train_features,
            feature_folds_dir / f"{fold_name}_train_features.parquet",
        )
        write_parquet(
            fold_val_features,
            feature_folds_dir / f"{fold_name}_val_features.parquet",
        )
        summary_rows.append(
            summarize_features(
                name=f"{fold_name}_train",
                df=fold_train_features,
                config=config,
            )
        )
        summary_rows.append(
            summarize_features(
                name=f"{fold_name}_val_backward_from_train",
                df=fold_val_features,
                config=config,
            )
        )

    summary = pd.DataFrame(summary_rows)
    summary.to_csv(paths["summary"], index=False)
    paths["config"].write_text(
        json.dumps(
            {
                **asdict(config),
                "split_dir": str(config.split_dir),
                "output_dir": str(config.output_dir),
                "category_cols_requested": list(config.categorical_cols),
                "rule": (
                    "split first; target lags/rolling use shift(1); "
                    "val/test use backward context only; categorical maps fit on train/development only"
                ),
            },
            ensure_ascii=False,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )
    paths["category_maps"].write_text(
        json.dumps(category_maps, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )

    print("Point-in-time feature engineering completed")
    print(f"split_dir : {config.split_dir}")
    print(f"output_dir: {config.output_dir}")
    print(f"lags      : {config.lags}")
    print(f"rolling   : {config.rolling_windows}")
    print(f"categorical_cols: {config.categorical_cols}")
    print("\nFeature summary:")
    print(summary.to_string(index=False))
    return paths


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Build point-in-time features from time-series split outputs."
    )
    parser.add_argument("--split-dir", type=Path, default=DEFAULT_SPLIT_DIR)
    parser.add_argument("--output-dir", type=Path, default=DEFAULT_OUTPUT_DIR)
    parser.add_argument("--version", default="v3")
    parser.add_argument("--expected-freq-minutes", type=int, default=15)
    parser.add_argument("--lags", type=parse_int_tuple, default=DEFAULT_LAGS)
    parser.add_argument(
        "--rolling-windows", type=parse_int_tuple, default=DEFAULT_ROLLING_WINDOWS
    )
    parser.add_argument(
        "--categorical-cols",
        type=parse_str_tuple,
        default=DEFAULT_CATEGORICAL_COLS,
        help=(
            "Comma-separated categorical columns to encode inside this step. "
            "Only columns existing in the data are used."
        ),
    )
    parser.add_argument("--timestamp-col", default=TIMESTAMP_COL)
    parser.add_argument("--site-col", default=SITE_COL)
    parser.add_argument("--target-col", default=TARGET_COL)
    return parser.parse_args()


def main() -> int:
    args = parse_args()
    config = TimeFeatureConfig(
        split_dir=args.split_dir.expanduser().resolve(),
        output_dir=args.output_dir.expanduser().resolve(),
        version=args.version,
        expected_freq_minutes=args.expected_freq_minutes,
        lags=args.lags,
        rolling_windows=args.rolling_windows,
        categorical_cols=args.categorical_cols,
        timestamp_col=args.timestamp_col,
        site_col=args.site_col,
        target_col=args.target_col,
    )
    run_feature_engineering(config)
    return 0


if __name__ == "__main__":
    # raise SystemExit(main())
    pass
